# PhenoBench Single-class (weed) Export — full frames (1024)

Exports a PhenoBench **TensorFlow Object Detection** bundle for the updated training pipeline (`agri_vision_edge.tfod_trainer`).

Partial (do-not-care) plants are handled per the official PhenoBench protocol (`plant_visibility <= 0.5`):

- **train.record / train_annotations** — partials are **dropped** (`include_partials=False`), so the model is never trained on partially-visible plants.
- **val / test / true_eval records + annotations** — partials are **carried but flagged** (`include_partials=True`, written as `ignore=1` / `is_partial` / `is_crowd`), so the `ave evaluate --ignore-partials` and trainer `eval_ignore_partials` knobs treat detections on them as do-not-care instead of false positives.

Generated artifacts: TFRecords, COCO annotations, label map, representative-dataset indices, and metadata.

## Environment

Kaggle runtime, Python 3.10. Pin the repo commit for reproducibility.

In [1]:
!python --version

Python 3.10.10


In [2]:
!pip install -q --no-cache-dir phenobench
!pip install -q --no-deps \
  git+https://github.com/frdiener/agri-vision-edge.git@abf74003b804d5c7130e368e0ae0eef696fa047e

## Configuration

In [3]:
from pathlib import Path
import json
from phenobench import PhenoBench

from agri_vision_edge.data import (
    PHENOBENCH_WEED_ONLY as DATASET_DEFINITION,
    split_indices,
    build_record,
    build_rep_indices,
    write_label_map,
    export_coco_annotations,
)
from agri_vision_edge.data.plant_boxes import PartialAwarePhenoBench

SEED = 42
# Store images at their NATIVE resolution (full 1024 frames / 512 tiles)
# and let the trainer's fixed_shape_resizer downsize to the model input
# at runtime. This is deliberate: TFOD's crop/zoom augmentations
# (random_crop_image, random_scale_crop_and_pad_to_square) run on the
# stored image BEFORE the resizer, so pre-downsampling here would starve
# them of detail.
IMAGE_SIZE = 1024

# Upstream do-not-care criterion: plant_visibility <= this ratio.
PARTIAL_THRESHOLD = 0.5

DATASET_ROOT = Path(
    "/kaggle/input/datasets/freimutdiener/phenobench-raw-dataset-v1-1-0/PhenoBench"
)

assert DATASET_ROOT.exists()

/opt/conda/lib/python3.10/site-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.16.5 and <1.23.0 is required for this version of SciPy (detected version 1.23.5
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


## Dataset Loading

We load semantics + instances + **plant_visibility** so partiality can be derived from the upstream visibility mask, and wrap the loader so `plant_bboxes` carry `visibility` / `is_partial`.

In [4]:
raw_train = PhenoBench(
    root=DATASET_ROOT,
    split="train",
    target_types=["semantics", "plant_instances", "plant_visibility"],
    ignore_partial=False,
)

raw_val = PhenoBench(
    root=DATASET_ROOT,
    split="val",
    target_types=["semantics", "plant_instances", "plant_visibility"],
    ignore_partial=False,
)

train_dataset = PartialAwarePhenoBench(
    raw_train, partial_threshold=PARTIAL_THRESHOLD
)
val_dataset = PartialAwarePhenoBench(
    raw_val, partial_threshold=PARTIAL_THRESHOLD
)

print("Train samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))

Train samples: 1407
Validation samples: 772


## Validation / Test Split

The official val split is halved into val + held-out test (deterministic seed).

In [5]:
val_idx, test_idx = split_indices(
    len(val_dataset),
    val_ratio=0.5,
    seed=SEED,
)

with open("val_test_split.json", "w") as f:
    json.dump({"val": val_idx, "test": test_idx}, f)

## TFRecord Export

`train.record` drops partials; the eval records keep them flagged do-not-care.

In [6]:
train_stats = build_record(
    "train.record",
    train_dataset,
    dataset_definition=DATASET_DEFINITION,
    target_size=IMAGE_SIZE,
    skip_negatives=False,
    include_partials=False,   # do not train on partials
)

true_eval_stats = build_record(
    "true_eval.record",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    target_size=IMAGE_SIZE,
    skip_negatives=False,
    include_partials=True,    # flag partials as do-not-care
)

val_stats = build_record(
    "val.record",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=val_idx,
    target_size=IMAGE_SIZE,
    skip_negatives=False,
    include_partials=True,
)

test_stats = build_record(
    "test.record",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=test_idx,
    target_size=IMAGE_SIZE,
    skip_negatives=False,
    include_partials=True,
)

train.record → written: 1407, 52 hard negatives included.
true_eval.record → written: 772, 34 hard negatives included.
val.record → written: 386, 16 hard negatives included.
test.record → written: 386, 18 hard negatives included.


## COCO Annotations

Used by `ave evaluate` (`--ignore-partials` reads the partial flags).

In [7]:
export_coco_annotations(
    "train_annotations.json",
    train_dataset,
    dataset_definition=DATASET_DEFINITION,
    include_partials=False,
)

export_coco_annotations(
    "true_eval_annotations.json",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    include_partials=True,
)

export_coco_annotations(
    "val_annotations.json",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=val_idx,
    include_partials=True,
)

export_coco_annotations(
    "test_annotations.json",
    val_dataset,
    dataset_definition=DATASET_DEFINITION,
    indices=test_idx,
    include_partials=True,
)

Wrote COCO annotations: train_annotations.json
Wrote COCO annotations: true_eval_annotations.json
Wrote COCO annotations: val_annotations.json
Wrote COCO annotations: test_annotations.json


{'images': 386, 'annotations': 1995, 'categories': 1}

## Representative Dataset (INT8 calibration)

In [8]:
rep_indices = build_rep_indices(
    dataset=train_dataset,
    num_samples=200,
    seed=SEED,
)

with open("rep_dataset.json", "w") as f:
    json.dump(rep_indices, f)

## Label Map

In [9]:
write_label_map(
    "label_map.pbtxt",
    dataset_definition=DATASET_DEFINITION,
)

Wrote label map: label_map.pbtxt


## Metadata

In [10]:
metadata = {
    "dataset_definition": {
        "name": DATASET_DEFINITION.name,
        "categories": DATASET_DEFINITION.categories,
    },
    "image_size": IMAGE_SIZE,
    "tiling": None,
    "partial_threshold": PARTIAL_THRESHOLD,
    "partials_policy": "drop in train, do-not-care in eval",
    "train_samples": len(train_dataset),
    "val_samples": len(val_dataset),
    "rep_samples": len(rep_indices),
    "split_seed": SEED,
    "train_stats": train_stats,
    "true_eval_stats": true_eval_stats,
    "val_stats": val_stats,
    "test_stats": test_stats,
}

with open("dataset_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

## Artifact Verification

In [11]:
artifacts = [
    "train.record", "val.record", "test.record", "true_eval.record",
    "train_annotations.json", "true_eval_annotations.json",
    "val_annotations.json", "test_annotations.json",
    "label_map.pbtxt", "rep_dataset.json",
    "val_test_split.json", "dataset_metadata.json",
]

missing = [p for p in artifacts if not Path(p).exists()]
assert not missing, f"Missing artifacts: {missing}"
print("All artifacts generated successfully.")

All artifacts generated successfully.


In [12]:
print(json.dumps(metadata, indent=2))

{
  "dataset_definition": {
    "name": "phenobench_weed_only",
    "categories": [
      {
        "id": 1,
        "name": "weed"
      }
    ]
  },
  "image_size": 1024,
  "tiling": null,
  "partial_threshold": 0.5,
  "partials_policy": "drop in train, do-not-care in eval",
  "train_samples": 1407,
  "val_samples": 772,
  "rep_samples": 200,
  "split_seed": 42,
  "train_stats": {
    "written": 1407,
    "target_size": 1024,
    "hard_negatives": 52
  },
  "true_eval_stats": {
    "written": 772,
    "target_size": 1024,
    "hard_negatives": 34
  },
  "val_stats": {
    "written": 386,
    "target_size": 1024,
    "hard_negatives": 16
  },
  "test_stats": {
    "written": 386,
    "target_size": 1024,
    "hard_negatives": 18
  }
}
